In [2]:

import pandas as pd
import numpy as np
import json

# Pancreas

In [5]:
organism = "homo_sapiens"
cell_source = "pancreas"
cell_state = None
reference = "hg19"
paper_doi = "https://doi.org/10.1016/j.cmet.2016.08.020"
table_link = "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5069352/bin/mmc2.xlsx"

# don't include in header
table_name = "paper_degs.xlsx"

header = [
    {
      "organism": organism,
      "cell_source": cell_source,
      "reference": reference,
      "paper_doi": paper_doi,
      "table_link": table_link
    }
]
  

## Get sheet names

In [3]:
metadata = pd.read_json("../metadata.json")
sheet_names = [obj["sheet_name"] for obj in metadata["tables"]]

sheet_names

['VariableGenes_Celltypes', 'ExpressedGenes_Celltypes']

## Read in files

In [6]:
excel = pd.read_excel(table_name, sheet_name = sheet_names, skiprows = 4)

In [7]:
for e in list(excel.values()):
    print(e.columns)

Index(['Rank', 'Unnamed: 1', 'α-cells', 'β-cells', 'γ-cells', 'δ-cells',
       'ε-cells', 'unclass endocrine', 'acinar cells', 'ductal cells',
       'MHC class II', 'mast cells', 'PSCs', 'endothelial cells',
       'Unnamed: 14', 'all cells', 'endocrine cells', 'exocrine cells'],
      dtype='object')
Index(['Rank', 'Unnamed: 1', 'α-cells', 'β-cells', 'γ-cells', 'δ-cells',
       'ε-cells', 'co-expression', 'unclass endocrine', 'acinar cells',
       'ductal cells', 'MHC class II', 'mast cells', 'PSCs',
       'endothelial cells', 'unclass exocrine'],
      dtype='object')


In [8]:
excel["ExpressedGenes_Celltypes"].columns

Index(['Rank', 'Unnamed: 1', 'α-cells', 'β-cells', 'γ-cells', 'δ-cells',
       'ε-cells', 'co-expression', 'unclass endocrine', 'acinar cells',
       'ductal cells', 'MHC class II', 'mast cells', 'PSCs',
       'endothelial cells', 'unclass exocrine'],
      dtype='object')

In [9]:
excel["VariableGenes_Celltypes"].columns

Index(['Rank', 'Unnamed: 1', 'α-cells', 'β-cells', 'γ-cells', 'δ-cells',
       'ε-cells', 'unclass endocrine', 'acinar cells', 'ductal cells',
       'MHC class II', 'mast cells', 'PSCs', 'endothelial cells',
       'Unnamed: 14', 'all cells', 'endocrine cells', 'exocrine cells'],
      dtype='object')

In [18]:
n_top_genes = 50

# VariableGenes_Celltypes: Lists with genes ranked in descending order according to biological variation within the different cell types.
# ExpressedGenes_Celltypes: Lists with genes ranked in descending order according to magnitude of expression for the different cell types (used in Figure 2B). 

# The file is sorted in descending order by most relevant genes (they did not release pvals or logfc)
df_first = excel["ExpressedGenes_Celltypes"].drop(
    columns=["Rank", "Unnamed: 1", "co-expression"]
    ).map(
        lambda x: x.replace("'", "")
    ).iloc[:n_top_genes].melt(
    ).rename(columns={"variable": "cell_type_label", "value": "feature_name"})
df_first["sheet_name"] = "ExpressedGenes_Celltypes"

df_second = excel["VariableGenes_Celltypes"].drop(
    columns=["Rank", "Unnamed: 1", "Unnamed: 14", "all cells"]
    ).map(
        lambda x: x.replace("'", "")
    ).iloc[:n_top_genes].melt(
    ).rename(columns={"variable": "cell_type_label", "value": "feature_name"})
df_second["sheet_name"] = "VariableGenes_Celltypes"


In [19]:
df = pd.concat([df_first, df_second], ignore_index= True)

df

,cell_type_label,feature_name,sheet_name
0,α-cells,GCG,ExpressedGenes_Celltypes
1,α-cells,TTR,ExpressedGenes_Celltypes
2,α-cells,B2M,ExpressedGenes_Celltypes
3,α-cells,CHGB,ExpressedGenes_Celltypes
4,α-cells,FTL,ExpressedGenes_Celltypes
...,...,...,...
1345,exocrine cells,PRSS3P2,VariableGenes_Celltypes
1346,exocrine cells,CELA2A,VariableGenes_Celltypes
1347,exocrine cells,REG1A,VariableGenes_Celltypes
1348,exocrine cells,MT2A,VariableGenes_Celltypes


## Unfiltered

In [20]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": None,
                "log_fc": None,
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'α-cells',
   'cell_source': 'pancreas',
   'cell_state': None,
   'feature_name': 'GCG',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'α-cells',
   'cell_source': 'pancreas',
   'cell_state': None,
   'feature_name': 'GCG',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'ExpressedGenes_Celltypes',
   'source_metrics': {'p_corr': None, 'log_fc': None}}}]

## Save unfiltered

In [21]:
with open("evidence_unfiltered_metrics.json", 'w') as f:
    json.dump(result, f, indent=4)

## Filtered

In [8]:
df.head()

,celltype,gene
0,α-cells,GCG
1,α-cells,TTR
2,α-cells,B2M
3,α-cells,CHGB
4,α-cells,FTL


In [9]:
df.celltype.value_counts()

celltype
α-cells              50
β-cells              50
γ-cells              50
δ-cells              50
ε-cells              50
unclass endocrine    50
acinar cells         50
ductal cells         50
MHC class II         50
mast cells           50
PSCs                 50
endothelial cells    50
unclass exocrine     50
Name: count, dtype: int64

In [10]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "celltype"
col_rank = "pct.1"

In [11]:
min_mean = 10
max_pval = 0.05
min_lfc = 1
max_gene_shares = 4

# filter by criteria
dfc = df # df.query(f"Marker == 1.0 & avg_logFC >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell type
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = m[col_ct].value_counts().min()

In [24]:
m[col_ct].value_counts()

celltype
endothelial cells    31
acinar cells         30
MHC class II         29
ductal cells         27
unclass exocrine     27
mast cells           22
ε-cells              17
PSCs                 17
α-cells              16
unclass endocrine    16
β-cells              14
δ-cells              11
γ-cells              10
Name: count, dtype: int64

In [13]:
# sample n_sample genes
markers = m.groupby("celltype").head(n_sample)
markers_dict = markers.groupby("celltype")["gene"].apply(lambda x: list(x)).to_dict()

In [25]:
markers.celltype.value_counts()

celltype
α-cells              10
β-cells              10
γ-cells              10
δ-cells              10
ε-cells              10
unclass endocrine    10
acinar cells         10
ductal cells         10
MHC class II         10
mast cells           10
PSCs                 10
endothelial cells    10
unclass exocrine     10
Name: count, dtype: int64

## Convert to evidence json



In [22]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = None

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'α-cells',
   'cell_source': 'pancreas',
   'cell_state': None,
   'feature_name': 'GCG',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'α-cells',
   'cell_source': 'pancreas',
   'cell_state': None,
   'feature_name': 'GCG',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000115263'},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'α-cells',
   'cell_source': 'pancreas',
   'cell_state': None,
   'feature_name': 'TM4SF4',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'α-cells',
   'cell_source': 'pancreas',
   'cell_state': None,
   'feature_name': 'TM4SF4',
   'feature_type': 'ensembl',
   'feature_identifier': 'ENSG00000169903'},
  'source': {'source_

## Save evidence

In [23]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 